<a href="https://colab.research.google.com/github/jabenitezz/Building-AI-Agents-for-Finance/blob/main/Chapter%201/chapter_1_lab_3_non-agentic_vs_agentic-workflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 1 - Lab 3: <font color='blue'>Agentic vs non-Agentic workfloww</font>

**<font color='purple'>Goal</font>**:
In this lab, you'll compare **Agentic** and **non-agentiw workflow**:

* Chatting with an LLM without access to external resources
* Chatting with an LLM with augmented information using external APIs:
    -  Fetch Historical Prices for a given stock (The same use case in the book)
    -  Collect News (Additional use case to practice)

**<font color='purple'>Tech stack</font>**:

We'll use :
* OpenAI Responses API ==> Standard calls + Function Callings
* OpenAI Agent SDK ==> Simple Agent Abstraction
* NewsApi: https://newsapi.org/

You need to create API Keys in both OpenAI (you need to add some credits) and NewsAPI (free until a certain # of calls).

Knowledge Cutoff of GPT-4.1 is June 2024

## Install packages

Install openai and newsapi packages

In [1]:
%pip install openai -q
%pip install newsapi-python -q

Add you OpenAI Key in Google colab: in the left vertical menu, you have an icon key. Click on it, and add your OpenAI key with a name

In [3]:
import os

try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    NEWS_API_KEY = userdata.get('NEWS_API_KEY')
except ImportError:
    # Running locally - read the keys from the environment (see ../.env.sample).
    OPENAI_API_KEY = os.environ['OPENAI_API_KEY']
    NEWS_API_KEY = os.environ['NEWS_API_KEY']

To have more information on how to call the API, you can read the documentation below.

In the various labs, you'll have a detailed overview on how to use the API.

https://platform.openai.com/docs/api-reference



# 1- Non Agentic Workflow

## Historical Price

### Responses API - GPT-4.1

https://platform.openai.com/docs/api-reference/responses

In [4]:
from openai import OpenAI

client = OpenAI(api_key = OPENAI_API_KEY)

response = client.responses.create(
  model="gpt-4.1",
  input="What is the last price of NVIDIA."
)

print(response)

Response(id='resp_0aee5d6c498a13cb006aa1a7683b6087d0a5a0c6d7ee8a7384', created_at=1788979048.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4.1-2025-04-14', object='response', output=[ResponseOutputMessage(id='msg_0aee5d6c498a13cb006aa1a768c42887d08cbe0187e263e713', content=[ResponseOutputText(annotations=[], text='As of the most recent market close on **June 14, 2024**, the last price of **NVIDIA (NVDA)** was **$131.88** per share.\n\nPlease note that stock prices fluctuate during trading hours. For the latest real-time price, refer to financial news sources or stock market platforms such as Yahoo Finance, Google Finance, or your brokerage account.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase=None)], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[], top_p=1.0, background=False, completed_at=1788979049.0, conversation=None, max_output_tokens=None, max_tool_calls=None, moderati

In [5]:
print(response.output[0].content[0].text)

As of the most recent market close on **June 14, 2024**, the last price of **NVIDIA (NVDA)** was **$131.88** per share.

Please note that stock prices fluctuate during trading hours. For the latest real-time price, refer to financial news sources or stock market platforms such as Yahoo Finance, Google Finance, or your brokerage account.


### Results

As yu can see, the LLM was not able to provide the last price of NVIDIA, as its knwoledge cufoff was on June 2024.

### Another Try

In this part, I'll provide a web site link. let's see what happens:

In [6]:
from openai import OpenAI

client = OpenAI(api_key = OPENAI_API_KEY)

response2 = client.responses.create(
  model="gpt-4.1",
  input="What is the last price of NVIDIA. Can you access to https://finance.yahoo.com/quote/NVDA to collect the last price? "
)

print(response2.output[0].content[0].text)

I can't browse the internet in real-time or access live data directly from web pages, including Yahoo Finance. However, you can easily find the latest price of NVIDIA (NVDA) by visiting [Yahoo Finance - NVIDIA](https://finance.yahoo.com/quote/NVDA) or by searching "NVIDIA stock price" on Google or your preferred financial news website.

If you need to know how to extract the price programmatically or want tips on tracking stock prices, feel free to ask!


I have the same issue, the LLM cannot provide the price because it cannot access and browse external websites like this.

## News

### Responses API

In [7]:
from openai import OpenAI

client = OpenAI(api_key = OPENAI_API_KEY)

response = client.responses.create(
  model="gpt-4.1",
  input="Give me the latest news about NVIDIA."
)

print(response.output[0].content[0].text)

Certainly! Here are the latest headlines and updates about NVIDIA as of June 2024:

### 1. **Record Growth and Market Cap**
NVIDIA continues its extraordinary growth trajectory, recently becoming the world’s most valuable chipmaker. Earlier this month (June 2024), NVIDIA’s market capitalization surged past $3 trillion, briefly making it the world’s second-most valuable publicly traded company, behind only Microsoft.

### 2. **GTC 2024 Announcements**
At the NVIDIA GTC 2024 event, several major product announcements were made, most notably:
- **Blackwell GPU Architecture**: NVIDIA unveiled its next-generation GPU architecture, "Blackwell," targeting AI and data center applications with significantly increased performance and efficiency over the previous Hopper generation.
- **Rubin AI Platform**: Announced as the next leap beyond Blackwell, expected in late 2025, focusing on accelerating generative AI workloads.
- **New AI Partnerships**: Cloud providers such as AWS, Google Cloud, and M

### Results

If you search for "NVIDIA Surpasses $3 Trillion Market Cap" in the web, you'll see that this news is from June 2024... as it's mentionned by the llm itself in the output.

# 2- Agentic Workflow:

In the following implementation, we'll use tools call to avoid knowledge cutoff and therefore build a basic agentic systeme

## Tools Calling without agent abstraction: Historical

Here, you'll learn how to use function calling with Responses API, without using any agent abstraction

1. You need to define what we call "Tools" object: Type (function), name, Description, parameters...
2. Declare your function: Tool to access external API
3. Call Responses API by enabling: "tools"

https://platform.openai.com/docs/guides/function-calling?api-mode=responses

In [8]:
import yfinance as yf

In [9]:
from openai import OpenAI
import json

client = OpenAI(api_key = OPENAI_API_KEY)

# 1. Define a list of callable tools for the model
tools = [
    {
        "type": "function",
        "name": "get_latest_stock_price",
        "description": "Fetch the current stock price for the given symbol.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {
                    "type": "string",
                    "description": "A ticker symbol of a given stock",
                },
            },
            "required": ["symbol"],
        },
    },
]

# Define the tool
def get_latest_stock_price(symbol: str) -> dict:
    """
    Fetch the current stock price for the given symbol.
    """

    ticker = yf.Ticker(symbol)
    hist = ticker.history(period="1d")
    latest_close = hist['Close'].iloc[-1]
    latest_date = hist.index[-1].strftime('%Y-%m-%d')

    return {'price': latest_close, 'date': latest_date}


input_list = [
    {"role": "user", "content": "What is the current price of nvidia."}
]

# 2. Prompt the model with tools defined
response = client.responses.create(
    model="gpt-4.1-mini",
    tools=tools,
    input=input_list,
)

print(response.output)

[ResponseFunctionToolCall(arguments='{"symbol":"NVDA"}', call_id='call_tkZJMnvDpsxK0VGvShTczVJg', name='get_latest_stock_price', type='function_call', id='fc_00ec677f0624797c006aa1a7e42d4487d08c5017d4bf3500e2', caller=None, namespace=None, status='completed')]


In [10]:
response.output

[ResponseFunctionToolCall(arguments='{"symbol":"NVDA"}', call_id='call_tkZJMnvDpsxK0VGvShTczVJg', name='get_latest_stock_price', type='function_call', id='fc_00ec677f0624797c006aa1a7e42d4487d08c5017d4bf3500e2', caller=None, namespace=None, status='completed')]

* This will generate a ResponseFunctionToolCall with the name of the function, and the argument (well interpreted by the model, it gives a symbol instead of the name of the company)
* We then to execute this function with the right argument


In [11]:
# Save function call outputs for subsequent requests
input_list += response.output

for item in response.output:
    if item.type == "function_call":
        if item.name == "get_latest_stock_price":
            print(json.loads(item.arguments))
            # 3. Execute the function logic
            prices = get_latest_stock_price(json.loads(item.arguments)['symbol'])
            print(prices)

            # 4. Provide function call results to the model
            input_list.append({
                "type": "function_call_output",
                "call_id": item.call_id,
                "output": json.dumps({
                  "prices": prices
                })
            })

print("Final input:")
print(input_list)

{'symbol': 'NVDA'}
{'price': np.float64(224.22000122070312), 'date': '2026-09-09'}
Final input:
[{'role': 'user', 'content': 'What is the current price of nvidia.'}, ResponseFunctionToolCall(arguments='{"symbol":"NVDA"}', call_id='call_tkZJMnvDpsxK0VGvShTczVJg', name='get_latest_stock_price', type='function_call', id='fc_00ec677f0624797c006aa1a7e42d4487d08c5017d4bf3500e2', caller=None, namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_tkZJMnvDpsxK0VGvShTczVJg', 'output': '{"prices": {"price": 224.22000122070312, "date": "2026-09-09"}}'}]


* Send back the whole result to the LLM to interpret and output the final answer

In [12]:
# Call the API with the updated input list
response = client.responses.create(
    model="gpt-4.1-mini",
    instructions="Respond only with price and date generated by a tool.",
    tools=tools,
    input=input_list,
)

# 5. The model should be able to give a response!
print("Final output:")
print(response.model_dump_json(indent=2))
print("\n" + response.output_text)

Final output:
{
  "id": "resp_00ec677f0624797c006aa1a83bdd9487d0abf89ae2a4863a13",
  "created_at": 1788979259.0,
  "error": null,
  "incomplete_details": null,
  "instructions": "Respond only with price and date generated by a tool.",
  "metadata": {},
  "model": "gpt-4.1-mini-2025-04-14",
  "object": "response",
  "output": [
    {
      "id": "msg_00ec677f0624797c006aa1a83c7e4087d0aaff8459e6293c63",
      "content": [
        {
          "annotations": [],
          "text": "The current price of Nvidia (NVDA) is $224.22 as of 2026-09-09.",
          "type": "output_text",
          "logprobs": []
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": null
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [
    {
      "name": "get_latest_stock_price",
      "parameters": {
        "type": "object",
        "properties": {
          "symbol": {
            "type": "string",
  

https://composio.dev/blog/claude-function-calling-tools

## OpenAI Agent SDK

In this part, we'll be using OpenAI Agent SDK.

You need first to install the package.

We'll create:
*  A tool: to fetch historical prices
*  An agent:
   - Describe the instructions to give to the agent
   - List of the tools the agent can use
   - Model

Install OpenAI Agents SDK

In [13]:
!pip install openai-agents -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 89.4 MB/s eta 0:00:00


In [14]:
import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

#We import the agent components
from agents import Agent, Runner, function_tool
import nest_asyncio
nest_asyncio.apply()

### Historical Prices Agent

#### Tool

➡ Create a tool to fetch historical prices

In [15]:
import yfinance as yf


@function_tool
def get_latest_stock_price(symbol: str) -> dict:
    """
    Fetch the current stock price for the given symbol.
    """

    ticker = yf.Ticker(symbol)
    hist = ticker.history(period="1d") # Fetch the last 1 day of historical data
    latest_close = hist['Close'].iloc[-1]
    latest_date = hist.index[-1]

    return {'price': latest_close, 'date': latest_date}

In [16]:
ticker = yf.Ticker("NVDA")
hist = ticker.history(period="1d")
hist

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2026-09-09 00:00:00-04:00,225.270004,226.179993,223.509995,224.145004,52267478,0.0,0.0


#### Agent

##### Define your agent

In [17]:
MODEL = "gpt-4.1"
stock_price_agent = Agent(
    name="Stock Price Agent",
    instructions="Fetch Historical prices for a given stock symbol",
    model = MODEL,
    tools=[get_latest_stock_price],
)

##### Run your agent

In [19]:


output = await Runner.run(
    starting_agent=stock_price_agent,
    input="What is today's date, and what is the current stock price of Nvidia?",
)

print(output.final_output)

Today's date is September 9, 2026. The current stock price of Nvidia (NVDA) is $224.07.


### News Agent

In [20]:
from newsapi import NewsApiClient
newsapi = NewsApiClient(api_key=NEWS_API_KEY)

from datetime import date, timedelta

#### Tool

Here I defined two tools:
* One fetching latest news from NewsAPI ==> **search_news**
* The other is summarizing these news, using an LLM ==> **summarize_news_api**

In [21]:
from openai import OpenAI
client = OpenAI(api_key = OPENAI_API_KEY)

In [22]:
END_DATE = date.today()
START_DATE = date.today()-timedelta(days=7)


def _search_news_internal(ticker: str, num_articles:int = 3, to_datetime:date = END_DATE,from_datetime:date = START_DATE) -> str:
  """
  Internal function that get the most recent news of a stock or an instrument

  Args:
  ticker (str): the stock ticker to be given to NEWSAPI
  num_articles (int): Number of news article to collect
  to_datetime (date): Today's date
  from_datetime (date): Date from which to collect the news (7 days before today for example)
  """



  all_articles = newsapi.get_everything(q=ticker,
                                            from_param=from_datetime,
                                            to=to_datetime,
                                            language='en',
                                            sort_by='relevancy',
                                            page_size=num_articles)

  news_concat =  [
    f"{article['title']}, {article['description']}, {article['publishedAt']},{article['content'][0:100]}"
    for article in all_articles['articles']
    ]

  return (".\n").join(news_concat)


@function_tool
def search_news(ticker: str, num_articles:int = 3, to_datetime:date = END_DATE,from_datetime:date = START_DATE) -> str:
  """
  Get the most recent news of a stock or an instrument

  Args:
  ticker (str): the stock ticker to be given to NEWSAPI
  num_articles (int): Number of news article to collect
  to_datetime (date): Today's date
  from_datetime (date): Date from which to collect the news (7 days before today for example)
  """
  return _search_news_internal(ticker, num_articles, to_datetime,from_datetime)

In [23]:

END_DATE = date.today()
START_DATE = date.today()-timedelta(days=7)

@function_tool
def summarize_news_api(ticker: str,num_articles:int = 3, to_datetime:date = END_DATE,from_datetime:date = START_DATE) -> str:
  """
  Summarize the news of a given stock or an instrument

  Args:
    ticker (str): the stock ticker to be given to NEWSAPI
    num_articles (int): Number of news article to collect
    to_datetime (date): Today's date
    from_datetime (date): Date from which to collect the news (7 days before today for example)

  """

  news = _search_news_internal(ticker, num_articles, to_datetime,from_datetime)

  prompt = f"Summarize the following text by extracting the key insights: {news}"

  response = client.responses.create(
  model="gpt-4.1",
  input= prompt
  )

  return response.output[0].content[0].text

#### Running NewsAPI itself without LLM/Agent

In [24]:
num_articles = 3
to_datetime = END_DATE
from_datetime = START_DATE
ticker = " NVDA"

all_articles = newsapi.get_everything(q=ticker,
                                            from_param=from_datetime,
                                            to=to_datetime,
                                            language='en',
                                            sort_by='relevancy',
                                            page_size=num_articles)

news_concat =  [
    f"{article['title']}, {article['description']}, {article['publishedAt']},{article['content'][0:100]}"
    for article in all_articles['articles']
    ]

news = (".\n").join(news_concat)

prompt = f"Summarize the following text by extracting the key insights: {news}"
response = client.responses.create(
  model="gpt-4.1",
  input= prompt
  )

print(response.output[0].content[0].text)

Here are the **key insights extracted** from the provided text:

1. **Advancement in Financial Trading Frameworks:**
   - *TradingAgents v0.4.0* was released by TauricResearch, featuring improvements such as look-ahead and point-in-time fixes, particularly across FRED market analyses, showcasing ongoing enhancement in multi-agent LLM-based trading systems.

2. **NVIDIA’s Strategic Moves:**
   - Jim Cramer discussed NVIDIA’s (NVDA) acquisition of Hugging Face, highlighting the company’s increased focus on open-source software infrastructure and its relevance in the context of AI and finance.

3. **NVIDIA’s Continued Strong Performance:**
   - Sustainable Growth Advisers (SGA) reported strong second-quarter 2026 results for NVIDIA, reinforcing a positive long-term outlook and the company's sustained growth trajectory within the U.S. Large Cap Growth Strategy.

**Summary:**  
There is rapid progress in multi-agent, LLM-powered financial trading frameworks (TradingAgents v0.4.0). Meanwhile

In [25]:
news

'Multi-Agents LLM Financial Trading Framework, TradingAgents: Multi-Agents LLM Financial Trading Framework - TauricResearch/TradingAgents, 2026-09-08T05:20:23Z,<ul><li>[2026-08] TradingAgents v0.4.0 released with look-ahead / point-in-time fixes across FRED ma.\nJim Cramer Discusses NVIDIA (NVDA) Acquiring Hugging Face, On the September 3 episode of Mad Money, Jim Cramer turned his attention to NVIDIA Corporation (NASDAQ:NVDA) and its expansion into open-source software infr..., 2026-09-07T20:22:49Z,On the September 3 episode of Mad Money, Jim Cramer turned his attention to NVIDIA Corporation (NASD.\nNvidia’s (NVDA) Strong Results Reinforce Long Term Outlook, Sustainable Growth Advisers (SGA), an investment management company, released its second-quarter 2026 investor letter for its “U.S. Large Cap Growth Strategy..., 2026-09-07T14:05:01Z,Sustainable Growth Advisers(SGA), an investment management company, released its second-quarter 2026'

In [26]:
#Another call to see if I have a consistent output
prompt = f"Summarize the following text by extracting the key insights: {news}"
response = client.responses.create(
  model="gpt-4.1",
  input= prompt
  )

In [27]:
print(response.output[0].content[0].text)

Here are the key insights extracted from the provided text:

1. **TradingAgents v0.4.0 Release** (August 2026)
   - The TradingAgents multi-agent LLM framework for financial trading has released version 0.4.0.
   - Update includes enhancements such as look-ahead and point-in-time fixes for handling FRED macroeconomic data.

2. **NVIDIA Expands into Open-Source AI** (September 2026)
   - On September 3, Jim Cramer discussed NVIDIA (NVDA) acquiring Hugging Face, signaling the company’s expansion into open-source software and AI infrastructure.

3. **NVIDIA’s Strong Financial Performance** (Q2 2026)
   - Sustainable Growth Advisers (SGA) highlighted NVIDIA’s strong quarterly results.
   - The firm believes this performance reinforces NVIDIA’s positive long-term outlook and its prospects for sustainable growth.


#### Agent: Fetch News

In [ ]:
from agents import Agent, Runner, function_tool

In [ ]:
MODEL = "gpt-4o-mini"
news_agent = Agent(
    name="News Agent",
    instructions="Get the latest news of a stock or an instrument using only the tools associated with the agent",
    model = MODEL,
    tools=[search_news],
)

In [ ]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="Give me the latest news about Apple.",
)

print(output.final_output)

Here are the latest news highlights about Apple:

1. **All-Time High Stock Price**:
   - Apple's stock price has recently hit an all-time high, currently sitting around $263 per share. This surge is attributed to soaring sales of the iPhone 17. *(Date: October 20, 2025)*

2. **Massive Returns for Stockholders**:
   - Over the past ten years, Apple stock has returned an incredible $847 billion to its investors through cash dividends and buybacks. *(Date: October 22, 2025)*

3. **Leading Earnings Reports**:
   - Apple is set to be a key player among several megacap hyperscalers reporting earnings, including Meta Platforms, Alphabet, Amazon, and Microsoft. Its stock remains close to a new buy point, showing positive early signs. *(Date: October 24, 2025)*

If you need more details or additional information, let me know!


In [ ]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="give me the latest news about Nvidia?",
)

print(output.final_output)

Here are the latest news articles about Nvidia (NVDA):

1. **Nvidia Stock at 8% of the S&P 500 Index Is a Big Problem for Investors. Let’s Do the Math.**  
   Published on: October 21, 2025  
   Summary: This article discusses how one single stock, Nvidia, significantly influences financial markets, much like how energy stocks once dominated the S&P 500 Index.

2. **Can Nvidia Stock Hit $300 in 2025?**  
   Published on: October 22, 2025  
   Summary: The article explores the potential for Nvidia’s stock to reach $300 by the end of 2025, fueled by its momentum in the artificial intelligence sector.

3. **NVIDIA Corporation (NVDA) And Taiwan Semiconductor Manufacturing (TSM) Produces the First Blackwell Wafer in US**  
   Published on: October 22, 2025  
   Summary: Nvidia announces its partnership with TSM to produce the first Blackwell wafer in the US, highlighting its growth as a leading revenue-generating stock.

If you need more information or specific articles, feel free to ask!


In [ ]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="Provide the latest news about Apple and Nvidia.",
)

print(output.final_output)

### Latest News on Apple (AAPL)

1. **Apple’s Stock Hits New All-Time High**  
   - **Date**: October 20, 2025  
   - Apple’s stock price recently reached an all-time high of approximately $263 per share, driven by surging iPhone 17 sales.

2. **Apple Stockholders Hit $850 Billion Jackpot**  
   - **Date**: October 22, 2025  
   - Over the past ten years, Apple stock has returned a substantial $847 billion to investors through dividends and buybacks.

3. **Apple Shines in Megacap Earnings Reports**  
   - **Date**: October 24, 2025  
   - Apple is part of a lineup of megacap hyperscalers making headlines in the latest earnings reports, holding near a fresh buy point.

### Latest News on Nvidia (NVDA)

1. **Nvidia Stock at 8% of the S&P 500 Index**  
   - **Date**: October 21, 2025  
   - The proportion of Nvidia stock in the S&P 500 Index poses challenges for investors, highlighting its significant market influence.

2. **Can Nvidia Stock Hit $300 in 2025?**  
   - **Date**: October 22

#### Agent: Summarize news

In [ ]:
MODEL = "gpt-4o-mini"
news_agent = Agent(
    name="News Agent",
    instructions="Get a summary about the latest news of a stock or an instrument using only the tools associated with the agent.",
    # instructions="Give me a summary about the latest news about OpenAI?",
    model = MODEL,
    tools=[summarize_news_api],
)

In [ ]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="Summarize the latest news of Apple?",
)

print(output.final_output)

Here are the latest insights regarding Apple (AAPL):

- **Stock Performance**: Apple’s stock has reached an all-time high of around $263 per share, attributed to strong sales of the iPhone 17.
  
- **Investor Returns**: Over the last decade, Apple has returned about $847 billion to investors through dividends and stock buybacks.

- **Analyst Ratings**: Wells Fargo has raised its price target for Apple to $290, reaffirming an “Overweight” rating and marking it as one of the top investment options among Fortune 500 companies as of late 2025.


In [ ]:
output = Runner.run_sync(
    starting_agent=news_agent,
    input="Give me a summary of the latest news of NVIDIA?",
)

print(output.final_output)

Here's a summary of the latest news regarding NVIDIA:

1. **Market Dominance**: Nvidia now makes up about 8% of the S&P 500 Index, raising systemic risks similar to when Exxon Mobil dominated the index. This concentration means the index's performance is heavily influenced by Nvidia, which could lead to volatility.

2. **AI-Driven Growth Expectations**: As a leader in the AI sector, there's speculation that Nvidia’s stock might exceed $300 by the end of 2025. The company's strong momentum in AI is crucial for its revenue and stock performance.

3. **Technological Advancements**: Nvidia, in collaboration with Taiwan Semiconductor Manufacturing Company (TSMC), has created the first Blackwell wafer in the US, highlighting its innovation and commitment to US manufacturing.

4. **Revenue Growth Leadership**: Nvidia continues to be recognized for exceptional revenue growth, with strong results and technological advancements reported as of October 2025.

Overall, while Nvidia's concentration 